# Fine-Tuning ViT Model untuk Emotion Recognition

**Tujuan:**
1. Fine-tune pre-trained ViT model dari HuggingFace
2. Handle class imbalance dengan teknik advanced
3. Optimasi untuk AMD RX 6600 LE GPU
4. Compare dengan CNN approach sebelumnya

**Model:** `trpakov/vit-face-expression`
**Dataset:** 6 emotion classes (angry, disgusted, happy, neutral, sad, surprised)

In [ ]:
# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# HuggingFace imports
from transformers import (
    ViTImageProcessor, 
    ViTForImageClassification,
    TrainingArguments, 
    Trainer,
    AutoConfig
)
from datasets import Dataset as HFDataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. 🔧 Setup AMD GPU dan DirectML

In [ ]:
# Setup device untuk AMD GPU
try:
    # Cek DirectML support
    if torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"🎮 Using GPU: {torch.cuda.get_device_name(0)}")
        print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    else:
        device = torch.device('cpu')
        print("💻 Using CPU (GPU not available)")
        
except Exception as e:
    print(f"⚠️ GPU setup issue: {e}")
    device = torch.device('cpu')
    print("💻 Fallback to CPU")

# Set memory optimization untuk AMD GPU
if device.type == 'cuda':
    torch.cuda.empty_cache()
    # Enable mixed precision untuk memory efficiency
    use_fp16 = True
    print("✅ GPU optimizations enabled")
else:
    use_fp16 = False
    print("⚠️ Running on CPU - training will be slower")

## 2. 📁 Load dan Analyze Dataset

In [ ]:
# Path configuration
BASE_PATH = 'D:/research/2025_iris_taufik/MultimodalEmoLearn-CNN-LSTM/data/'
MODEL_SAVE_PATH = 'D:/research/2025_iris_taufik/MultimodalEmoLearn-CNN-LSTM/models/'

print("📁 Loading dataset...")

try:
    # Load image data
    X_train_images = np.load(BASE_PATH + 'images/X_train_images.npy')
    X_val_images = np.load(BASE_PATH + 'images/X_val_images.npy')
    X_test_images = np.load(BASE_PATH + 'images/X_test_images.npy')
    
    # Load labels
    y_train = np.load(BASE_PATH + 'images/y_train_images.npy')
    y_val = np.load(BASE_PATH + 'images/y_val_images.npy')
    y_test = np.load(BASE_PATH + 'images/y_test_images.npy')
    
    print(f"✅ Dataset loaded successfully!")
    print(f"Training: {X_train_images.shape[0]:,} samples")
    print(f"Validation: {X_val_images.shape[0]:,} samples")
    print(f"Test: {X_test_images.shape[0]:,} samples")
    print(f"Image shape: {X_train_images.shape[1:]}")
    
except FileNotFoundError as e:
    print(f"❌ Error loading data: {e}")
    print("💡 Please ensure data files exist in the correct path")

In [ ]:
# Analyze class distribution
unique_labels = np.unique(y_train)
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
num_classes = len(unique_labels)

print(f"🏷️ Classes ({num_classes}): {list(unique_labels)}")
print(f"📊 Label mapping: {label_to_id}")

# Convert string labels to integers
y_train_ids = np.array([label_to_id[label] for label in y_train])
y_val_ids = np.array([label_to_id[label] for label in y_val])
y_test_ids = np.array([label_to_id[label] for label in y_test])

# Analyze class imbalance
from collections import Counter
train_distribution = Counter(y_train)
print(f"\n📊 Training Distribution:")
for emotion, count in train_distribution.items():
    percentage = (count / len(y_train)) * 100
    print(f"   {emotion}: {count:,} ({percentage:.1f}%)")

# Calculate imbalance ratio
max_count = max(train_distribution.values())
min_count = min(train_distribution.values())
imbalance_ratio = max_count / min_count
print(f"\n⚖️ Imbalance ratio: {imbalance_ratio:.1f}:1")
if imbalance_ratio > 10:
    print("⚠️ SEVERE class imbalance detected! Will use advanced techniques.")
elif imbalance_ratio > 5:
    print("⚠️ Significant class imbalance detected.")
else:
    print("✅ Moderate class imbalance.")

## 3. 🔄 Data Preprocessing untuk ViT

In [ ]:
# Load ViT processor
MODEL_NAME = "trpakov/vit-face-expression"

print(f"🔄 Loading ViT processor from {MODEL_NAME}...")
try:
    processor = ViTImageProcessor.from_pretrained(MODEL_NAME)
    print("✅ Processor loaded successfully")
    print(f"Expected image size: {processor.size}")
    print(f"Normalization: mean={processor.image_mean}, std={processor.image_std}")
except Exception as e:
    print(f"❌ Error loading processor: {e}")
    print("💡 Using default ViT processor")
    processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

In [ ]:
class EmotionDataset(Dataset):
    """Custom dataset untuk emotion recognition"""
    
    def __init__(self, images, labels, processor, augment=False):
        self.images = images
        self.labels = labels
        self.processor = processor
        self.augment = augment
        
        # Simple augmentation untuk training
        if augment:
            from torchvision import transforms
            self.augment_transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=5),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05)
            ])
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Get image dan convert ke PIL
        image = self.images[idx]
        
        # Normalize jika perlu (dari 0-1 ke 0-255)
        if image.max() <= 1.0:
            image = (image * 255).astype(np.uint8)
        
        # Convert ke PIL Image
        pil_image = Image.fromarray(image)
        
        # Apply augmentation jika training
        if self.augment:
            pil_image = self.augment_transform(pil_image)
        
        # Process dengan ViT processor
        processed = self.processor(pil_image, return_tensors="pt")
        
        return {
            'pixel_values': processed['pixel_values'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
print("📦 Creating datasets...")
train_dataset = EmotionDataset(X_train_images, y_train_ids, processor, augment=True)
val_dataset = EmotionDataset(X_val_images, y_val_ids, processor, augment=False)
test_dataset = EmotionDataset(X_test_images, y_test_ids, processor, augment=False)

print(f"✅ Datasets created:")
print(f"   Train: {len(train_dataset)} samples (with augmentation)")
print(f"   Validation: {len(val_dataset)} samples")
print(f"   Test: {len(test_dataset)} samples")

## 4. ⚖️ Handle Class Imbalance

In [ ]:
# Calculate class weights untuk loss function
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train_ids),
    y=y_train_ids
)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f"📊 Class weights for balanced loss:")
for i, (emotion, weight) in enumerate(zip(unique_labels, class_weights)):
    print(f"   {emotion}: {weight:.3f}")

# Create weighted sampler untuk training
def create_weighted_sampler(labels):
    """Create weighted sampler untuk balanced sampling"""
    class_counts = np.bincount(labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(labels),
        replacement=True
    )

train_sampler = create_weighted_sampler(y_train_ids)
print("✅ Weighted sampler created for balanced training")

## 5. 🤖 Load dan Configure ViT Model

In [ ]:
# Load pre-trained model
print(f"🤖 Loading pre-trained model: {MODEL_NAME}")

try:
    # Load config dan check original classes
    config = AutoConfig.from_pretrained(MODEL_NAME)
    original_num_labels = config.num_labels if hasattr(config, 'num_labels') else None
    
    print(f"Original model classes: {original_num_labels}")
    print(f"Our dataset classes: {num_classes}")
    
    # Update config untuk our classes
    config.num_labels = num_classes
    config.id2label = id_to_label
    config.label2id = label_to_id
    
    # Load model dengan updated config
    model = ViTForImageClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        ignore_mismatched_sizes=True  # Ignore classifier head size mismatch
    )
    
    print("✅ Model loaded successfully")
    print(f"Model size: {sum(p.numel() for p in model.parameters()):,} parameters")
    print(f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("💡 Using base ViT model instead")
    model = ViTForImageClassification.from_pretrained(
        "google/vit-base-patch16-224",
        num_labels=num_classes,
        id2label=id_to_label,
        label2id=label_to_id
    )

# Move model ke device
model.to(device)
print(f"🎮 Model moved to {device}")

## 6. 🎯 Custom Loss Function untuk Imbalanced Data

In [ ]:
class WeightedCrossEntropyLoss(nn.Module):
    """Weighted Cross Entropy Loss untuk handle class imbalance"""
    
    def __init__(self, class_weights):
        super().__init__()
        self.class_weights = class_weights
        self.loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    
    def forward(self, logits, labels):
        return self.loss_fn(logits, labels)

class FocalLoss(nn.Module):
    """Focal Loss untuk extreme class imbalance"""
    
    def __init__(self, alpha=1, gamma=2, class_weights=None):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.class_weights = class_weights
    
    def forward(self, logits, labels):
        ce_loss = nn.CrossEntropyLoss(weight=self.class_weights, reduction='none')(logits, labels)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# Choose loss function based on imbalance severity
if imbalance_ratio > 10:
    print("🎯 Using Focal Loss for severe imbalance")
    criterion = FocalLoss(alpha=1, gamma=2, class_weights=class_weights_tensor)
else:
    print("🎯 Using Weighted Cross Entropy Loss")
    criterion = WeightedCrossEntropyLoss(class_weights_tensor)

print("✅ Custom loss function configured")

## 7. 🏃‍♂️ Custom Trainer untuk Advanced Techniques

In [ ]:
class CustomTrainer(Trainer):
    """Custom trainer dengan advanced techniques untuk imbalanced data"""
    
    def __init__(self, *args, custom_loss_fn=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_loss_fn = custom_loss_fn
    
    def compute_loss(self, model, inputs, return_outputs=False):
        """Custom loss computation"""
        labels = inputs.get("labels")
        outputs = model(**inputs)
        
        if self.custom_loss_fn is not None:
            loss = self.custom_loss_fn(outputs.logits, labels)
        else:
            loss = outputs.loss
            
        return (loss, outputs) if return_outputs else loss
    
    def get_train_dataloader(self):
        """Custom dataloader dengan weighted sampling"""
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
            
        return DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=train_sampler,
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

def compute_metrics(eval_pred):
    """Compute metrics untuk evaluation"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Overall accuracy
    accuracy = accuracy_score(labels, predictions)
    
    # Per-class metrics
    report = classification_report(labels, predictions, output_dict=True, zero_division=0)
    
    # Balanced accuracy (important for imbalanced data)
    from sklearn.metrics import balanced_accuracy_score
    balanced_acc = balanced_accuracy_score(labels, predictions)
    
    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'macro_f1': report['macro avg']['f1-score'],
        'weighted_f1': report['weighted avg']['f1-score']
    }

print("✅ Custom trainer dan metrics configured")

## 8. 🚀 Training Configuration dan Fine-Tuning

In [ ]:
# Training arguments optimized for AMD GPU
training_args = TrainingArguments(
    output_dir=os.path.join(MODEL_SAVE_PATH, 'vit-emotion-finetuned'),
    num_train_epochs=20,  # Start with reasonable epochs
    per_device_train_batch_size=8 if device.type == 'cuda' else 4,  # Conservative for GPU memory
    per_device_eval_batch_size=16 if device.type == 'cuda' else 8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=os.path.join(MODEL_SAVE_PATH, 'logs'),
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="balanced_accuracy",  # Use balanced accuracy untuk imbalanced data
    greater_is_better=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    push_to_hub=False,
    report_to="none",  # Disable wandb/tensorboard for simplicity
    fp16=use_fp16,  # Mixed precision untuk memory efficiency
    dataloader_pin_memory=True,
    gradient_accumulation_steps=2,  # Simulate larger batch size
    learning_rate=2e-5,  # Conservative learning rate untuk fine-tuning
    lr_scheduler_type="cosine",
    seed=42
)

print(f"🚀 Training configuration:")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Learning rate: {training_args.learning_rate}")
print(f"   FP16: {training_args.fp16}")
print(f"   Device: {device}")

In [ ]:
# Initialize trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    custom_loss_fn=criterion
)

print("✅ Trainer initialized with custom loss and weighted sampling")

# Print model info sebelum training
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Information:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Trainable ratio: {trainable_params/total_params:.1%}")

In [ ]:
# Start fine-tuning
print("🏃‍♂️ Starting fine-tuning...")
print("=" * 50)

import time
start_time = time.time()

try:
    # Train model
    trainer.train()
    
    training_time = time.time() - start_time
    print(f"\n✅ Training completed in {training_time/3600:.2f} hours")
    
    # Save final model
    trainer.save_model(os.path.join(MODEL_SAVE_PATH, 'vit-emotion-final'))
    processor.save_pretrained(os.path.join(MODEL_SAVE_PATH, 'vit-emotion-final'))
    
    print(f"💾 Model saved to: {os.path.join(MODEL_SAVE_PATH, 'vit-emotion-final')}")
    
except Exception as e:
    print(f"❌ Training failed: {e}")
    print("💡 Try reducing batch size or using CPU")

## 9. 📊 Evaluation dan Analysis

In [ ]:
# Evaluate on test set
print("📊 Evaluating on test set...")

try:
    # Get predictions
    test_results = trainer.predict(test_dataset)
    test_predictions = np.argmax(test_results.predictions, axis=1)
    test_labels = test_results.label_ids
    
    # Calculate metrics
    test_accuracy = accuracy_score(test_labels, test_predictions)
    
    from sklearn.metrics import balanced_accuracy_score, f1_score
    test_balanced_acc = balanced_accuracy_score(test_labels, test_predictions)
    test_f1_macro = f1_score(test_labels, test_predictions, average='macro')
    test_f1_weighted = f1_score(test_labels, test_predictions, average='weighted')
    
    print(f"\n🎯 Test Results:")
    print(f"   Accuracy: {test_accuracy:.4f}")
    print(f"   Balanced Accuracy: {test_balanced_acc:.4f}")
    print(f"   F1-Score (Macro): {test_f1_macro:.4f}")
    print(f"   F1-Score (Weighted): {test_f1_weighted:.4f}")
    
    # Detailed classification report
    print(f"\n📈 Classification Report:")
    print(classification_report(test_labels, test_predictions, target_names=unique_labels))
    
except Exception as e:
    print(f"❌ Evaluation failed: {e}")

In [ ]:
# Create confusion matrix
if 'test_predictions' in locals():
    cm = confusion_matrix(test_labels, test_predictions)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=unique_labels, yticklabels=unique_labels)
    plt.title(f'ViT Fine-tuned Model - Confusion Matrix\nTest Accuracy: {test_accuracy:.4f}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_PATH, 'vit_confusion_matrix.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    # Per-class accuracy
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(unique_labels, per_class_acc, alpha=0.7, color='skyblue', edgecolor='black')
    plt.title('Per-Class Accuracy - ViT Fine-tuned Model')
    plt.ylabel('Accuracy')
    plt.xlabel('Emotion Class')
    plt.xticks(rotation=45)
    plt.ylim(0, 1.05)
    
    # Add value labels on bars
    for bar, acc in zip(bars, per_class_acc):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{acc:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_PATH, 'vit_per_class_accuracy.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n📊 Per-class Accuracy:")
    for emotion, acc in zip(unique_labels, per_class_acc):
        print(f"   {emotion}: {acc:.4f}")

## 10. 🔄 Compare dengan Baseline CNN

In [ ]:
# Load baseline results jika ada
try:
    import pickle
    
    # Coba load hasil CNN sebelumnya
    with open(os.path.join(MODEL_SAVE_PATH, 'cnn_training_info.pkl'), 'rb') as f:
        cnn_results = pickle.load(f)
    
    print(f"📊 Comparison with Baseline CNN:")
    print(f"   CNN Accuracy: {cnn_results.get('test_accuracy', 'N/A'):.4f}")
    print(f"   ViT Accuracy: {test_accuracy:.4f}")
    
    if 'test_accuracy' in cnn_results:
        improvement = test_accuracy - cnn_results['test_accuracy']
        print(f"   Improvement: {improvement:.4f} ({improvement*100:.2f}%)")
        
        if improvement > 0:
            print("   🎉 ViT outperforms CNN!")
        else:
            print("   📝 CNN still competitive")
    
except FileNotFoundError:
    print("⚠️ Baseline CNN results not found")
    print("💡 Run CNN training first untuk comparison")

## 11. 💾 Save Results dan Model Info

In [ ]:
# Save comprehensive results
if 'test_accuracy' in locals():
    vit_results = {
        'model_name': MODEL_NAME,
        'model_type': 'ViT_Fine_tuned',
        'num_classes': num_classes,
        'class_names': list(unique_labels),
        'label_mapping': label_to_id,
        'test_accuracy': test_accuracy,
        'balanced_accuracy': test_balanced_acc,
        'f1_macro': test_f1_macro,
        'f1_weighted': test_f1_weighted,
        'per_class_accuracy': dict(zip(unique_labels, per_class_acc)),
        'training_config': {
            'epochs': training_args.num_train_epochs,
            'batch_size': training_args.per_device_train_batch_size,
            'learning_rate': training_args.learning_rate,
            'fp16': training_args.fp16,
            'loss_function': type(criterion).__name__,
            'weighted_sampling': True,
            'class_imbalance_ratio': imbalance_ratio
        },
        'device_info': str(device),
        'training_time_hours': training_time / 3600 if 'training_time' in locals() else None,
        'total_parameters': total_params,
        'trainable_parameters': trainable_params
    }
    
    # Save results
    with open(os.path.join(MODEL_SAVE_PATH, 'vit_fine_tuned_results.pkl'), 'wb') as f:
        pickle.dump(vit_results, f)
    
    print(f"💾 Results saved to: {os.path.join(MODEL_SAVE_PATH, 'vit_fine_tuned_results.pkl')}")
    
    # Create summary report
    summary_text = f"""
ViT FINE-TUNING SUMMARY REPORT
===============================

Model: {MODEL_NAME}
Dataset: {len(unique_labels)} emotion classes ({len(y_train)} train, {len(y_val)} val, {len(y_test)} test)
Class Imbalance Ratio: {imbalance_ratio:.1f}:1

PERFORMANCE METRICS:
- Test Accuracy: {test_accuracy:.4f}
- Balanced Accuracy: {test_balanced_acc:.4f}
- F1-Score (Macro): {test_f1_macro:.4f}
- F1-Score (Weighted): {test_f1_weighted:.4f}

TRAINING CONFIGURATION:
- Loss Function: {type(criterion).__name__}
- Weighted Sampling: Yes
- Learning Rate: {training_args.learning_rate}
- Batch Size: {training_args.per_device_train_batch_size}
- Epochs: {training_args.num_train_epochs}
- Mixed Precision (FP16): {training_args.fp16}
- Device: {device}

MODEL INFO:
- Total Parameters: {total_params:,}
- Trainable Parameters: {trainable_params:,}
- Training Time: {training_time/3600:.2f} hours

PER-CLASS ACCURACY:
"""
    
    for emotion, acc in zip(unique_labels, per_class_acc):
        summary_text += f"- {emotion}: {acc:.4f}\n"
    
    # Save summary
    with open(os.path.join(MODEL_SAVE_PATH, 'vit_fine_tuning_summary.txt'), 'w') as f:
        f.write(summary_text)
    
    print(f"📄 Summary report saved to: {os.path.join(MODEL_SAVE_PATH, 'vit_fine_tuning_summary.txt')}")

## 12. 🎯 Final Summary dan Recommendations

In [ ]:
print("=" * 60)
print("🎉 ViT FINE-TUNING COMPLETED!")
print("=" * 60)

if 'test_accuracy' in locals():
    print(f"\n📊 FINAL RESULTS:")
    print(f"   🎯 Test Accuracy: {test_accuracy:.4f}")
    print(f"   ⚖️ Balanced Accuracy: {test_balanced_acc:.4f}")
    print(f"   📈 F1-Score (Macro): {test_f1_macro:.4f}")
    
    print(f"\n🛠️ TECHNIQUES USED:")
    print(f"   ✅ Pre-trained ViT model fine-tuning")
    print(f"   ✅ Weighted loss function untuk class imbalance")
    print(f"   ✅ Weighted sampling untuk balanced training")
    print(f"   ✅ Data augmentation")
    print(f"   ✅ Mixed precision training (FP16)")
    print(f"   ✅ Custom trainer dengan advanced metrics")
    
    print(f"\n📝 RECOMMENDATIONS:")
    if test_balanced_acc > 0.7:
        print(f"   🎉 Excellent results! Model ready for deployment")
    elif test_balanced_acc > 0.6:
        print(f"   ✅ Good results. Consider ensemble dengan landmark model")
    else:
        print(f"   ⚠️ Results need improvement. Consider:")
        print(f"      - More data augmentation")
        print(f"      - Different loss functions (Focal Loss)")
        print(f"      - Ensemble methods")
        print(f"      - Additional training data")
    
    print(f"\n🔄 NEXT STEPS:")
    print(f"   1. Integrate dengan landmark model untuk fusion")
    print(f"   2. Test pada real-world data")
    print(f"   3. Deploy untuk production use")
    print(f"   4. Monitor performance dan retrain jika perlu")
    
else:
    print("❌ Training was not completed successfully")
    print("💡 Please check the error messages above")

print("\n" + "=" * 60)